# ClimaCity Paris -- Jour 1
## Exploration batch : l'API RDD et l'API DataFrame

**Module** : Traitement de donnees massives avec Apache Spark et PySpark  
**Duree** : 1 journee (6 heures effectives)  
**Prerequis** : Python intermediaire, notions de Pandas et NumPy

---

Ce notebook couvre l'integralite du Jour 1 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : l'API bas niveau RDD, la semantique de l'evaluation paresseuse, les transformations et actions fondamentales, et une premiere lecture du Spark UI.
- **Partie 2 -- Apres-midi (3 h)** : l'API haut niveau DataFrame, le nettoyage des donnees, la jointure avec les observations meteorologiques, la persistance en memoire et l'ecriture en Parquet partitionne.

> **Convention dans ce notebook**  
> Les cellules marquees `# [EXERCICE]` contiennent une consigne. Vous devez completer le code avant de passer a la cellule suivante.  
> Les cellules marquees `# [CORRECTION]` proposent une solution possible -- ne les regardez qu'apres avoir tente.


---
## Section 0 -- Configuration de l'environnement

Toutes les constantes du projet sont declarees ici. Adaptez les chemins si necessaire,
puis executez cette cellule avant toute autre.


In [39]:
import os
import sys
import time
import platform
import subprocess
from pathlib import Path

# ── Mac Apple Silicon : Java arm64 + workers PySpark arm64 ───────────────────
# brew install openjdk@17 installe un JDK keg-only : java_home -v 17 ne le voit
# pas tant qu'il n'est pas symlinké. On pointe JAVA_HOME explicitement.
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"

    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

    _java_ver = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print(f"[Java] JAVA_HOME={os.environ.get('JAVA_HOME', '(non défini)')}")
    print(f"[Java] {(_java_ver.stderr or _java_ver.stdout).splitlines()[0]}")
    print(f"[PySpark] PYSPARK_PYTHON={os.environ['PYSPARK_PYTHON']}")

# ── Chemins de donnees ──────────────────────────────────────────────────────
# "data" = dossier du projet ; "../data" = fallback si vous lancez depuis ailleurs
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
VELIB_RAW_DIR  = DATA_DIR / "velib" / "raw"              # CSV bruts (RDD)
HISTORIQUE_STATIONS_CSV = next(
    (p for p in [
        Path("historique_stations.csv"),
        DATA_DIR / "velib" / "raw" / "historique_stations.csv",
    ] if p.exists()),
    Path("historique_stations.csv"),
)  # historique Velib (lovasoa)
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"          # Parquet partitionne
STATIONS_CSV   = DATA_DIR / "velib" / "stations_info.csv"
METEO_CSV      = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR     = DATA_DIR / "output"                     # ecriture des resultats

# ── Parametres Spark ────────────────────────────────────────────────────────
APP_NAME       = "ClimaCity-Paris"
MASTER         = "local[*]"                              # toutes les CPU locales
#nombre de coueur du processeur alloué à la partition
SHUFFLE_PARTS  = 8                                       # adapte aux volumes du cours
SEED           = 42

# Verification des chemins
for p in [VELIB_RAW_DIR, VELIB_PARQ_DIR, STATIONS_CSV, METEO_CSV, HISTORIQUE_STATIONS_CSV]:
    status = "OK" if p.exists() else "MANQUANT"
    print(f"[{status}] {p}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] {OUTPUT_DIR} (cree si absent)")


[Java] JAVA_HOME=/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
[Java] openjdk version "17.0.19" 2026-04-21
[PySpark] PYSPARK_PYTHON=/Users/romain/Desktop/SparkVelib/.venv-spark/bin/python-arm64
[OK] data/velib/raw
[OK] data/velib/parquet
[OK] data/velib/stations_info.csv
[OK] data/meteo/paris_montsouris_horaire.csv
[OK] historique_stations.csv
[OK] data/output (cree si absent)


---
## Section 0.1 -- Collecte des données Vélib'

Deux sources complémentaires sont nécessaires :

- **Les informations statiques des stations** (nom, coordonnées, capacité) : publiées
  en temps réel par l'API GBFS de Vélib' Métropole, sans authentification.
- **L'historique de disponibilité** : fichier local `historique_stations.csv`
  (projet [lovasoa/historique-velib-opendata](https://github.com/lovasoa/historique-velib-opendata)),
  snapshots réguliers de l'état du réseau.

Exécutez les cellules ci-dessous **une seule fois** avant le démarrage de la session
Spark. Placez `historique_stations.csv` à la **racine du projet** (ou dans `data/velib/raw/`).

In [40]:
import requests
import json
import pandas as pd

# ── Endpoint GBFS Vélib' Métropole (sans clé, sans inscription) ──────────────
GBFS_STATION_INFO = (
    "https://velib-metropole-opendata.smovengo.cloud"
    "/opendata/Velib_Metropole/station_information.json"
)

print("Téléchargement des informations de stations...")
resp = requests.get(GBFS_STATION_INFO, timeout=15)
resp.raise_for_status()

stations = resp.json()["data"]["stations"]
print(f"  {len(stations)} stations reçues")

df_stations = pd.DataFrame(stations)[[
    "station_id", "name", "lat", "lon", "capacity", "stationCode"
]]

# Normalisation : station_id en entier (certaines valeurs sont des strings)
df_stations["station_id"] = df_stations["station_id"].astype(int)

# Extraction de l'arrondissement depuis le nom de la station
# Convention Vélib' : le code commune figure dans station_id (75xxx = Paris)
df_stations["code_arr"] = df_stations["station_id"] // 1000

STATIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_stations.to_csv(STATIONS_CSV, index=False, sep=";")
print(f"  Sauvegardé : {STATIONS_CSV}")
print(f"  Capacité totale du réseau : {df_stations['capacity'].sum():,} bornettes")
df_stations.head(5)

Téléchargement des informations de stations...
  1517 stations reçues
  Sauvegardé : data/velib/stations_info.csv
  Capacité totale du réseau : 49,005 bornettes


,station_id,name,lat,lon,capacity,stationCode,code_arr
0,213688169,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,16107,213688
1,19179944124,Hôpital Mondor,48.798922,2.453745,28,40001,19179944
2,18795078746,Charcot - Benfleet,48.878370,2.440524,28,32304,18795078
3,36255,Toudouze - Clauzel,48.879296,2.337360,21,9020,36
4,251039991,Cassini - Denfert-Rochereau,48.837526,2.336035,25,14111,251039


In [41]:
# ── Historique Velib (lovasoa) ───────────────────────────────────────────────
# Format CSV sans en-tête :
#   date,capacity,available_mechanical,available_electrical,station_name,station_geo,operative
# Exemple :
#   2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True

if not HISTORIQUE_STATIONS_CSV.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {HISTORIQUE_STATIONS_CSV}\n"
        "Placez historique_stations.csv à la racine du projet "
        "ou dans data/velib/raw/."
    )

taille_mb = HISTORIQUE_STATIONS_CSV.stat().st_size / 1_048_576
print(f"Fichier historique : {HISTORIQUE_STATIONS_CSV.resolve()}")
print(f"  Taille           : {taille_mb:.0f} MB")

print("\nAperçu (3 premières lignes) :")
with open(HISTORIQUE_STATIONS_CSV, encoding="utf-8") as f:
    for i, ligne in enumerate(f):
        if i >= 3:
            break
        print(f"  {ligne.rstrip()}")

print("\n[OK] Données historiques prêtes — chargement via sc.textFile() dans la Partie 1.")

Fichier historique : /Users/romain/Desktop/SparkVelib/historique_stations.csv
  Taille           : 376 MB

Aperçu (3 premières lignes) :
  2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
  2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
  2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True

[OK] Données historiques prêtes — chargement via sc.textFile() dans la Partie 1.


In [42]:
# Pas de téléchargement : le fichier local est utilisé directement par Spark.
print(f"Source RDD : {HISTORIQUE_STATIONS_CSV.resolve()}")

Source RDD : /Users/romain/Desktop/SparkVelib/historique_stations.csv


# **Nota bene**
- `historique_stations.csv` fait ~400 Mo et contient plusieurs millions de lignes — le premier `count()` Spark peut prendre une minute.
- Le dépôt GitHub ne publie plus les fichiers `.csv.gz` par quinzaine ; le CSV consolidé local remplace ce téléchargement.


---
# PARTIE 1 -- L'API RDD (matin)

## 1.1 La SparkSession et le SparkContext

### Pourquoi Spark ?

Pandas est un outil remarquable pour les donnees qui tiennent en memoire sur une seule
machine. Mais il atteint ses limites lorsque le volume depasse quelques gigaoctets, ou
lorsque le traitement doit s'executer en parallele sur plusieurs coeurs ou plusieurs noeuds.

Apache Spark repond a ce besoin en proposant un modele de calcul distribue et tolerant aux
pannes, dans lequel les donnees sont representees comme des collections immuables et
partitionnees, les **RDD** (*Resilient Distributed Datasets*).

Meme en mode local -- sur une seule machine, comme c'est le cas dans ce cours -- Spark
parallelise les traitements sur tous les coeurs disponibles et permet de travailler sur
des volumes qui depassent la memoire vive grace a la gestion automatique du debordement
sur disque.

### Le point d'entree : `SparkSession`

Depuis Spark 2.0, `SparkSession` est le point d'entree unifie. Il encapsule le
`SparkContext` (execution RDD), le `SQLContext` (SQL et DataFrame) et le
`HiveContext`. On n'en cree qu'une par application.


### Reset Spark (si erreur Py4J / Connection refused)

1. **Kernel → Restart Kernel**
2. Exécutez Section 0, puis la cellule ci-dessous
3. Puis la cellule **Exercice 1** (SparkSession)


In [43]:
from pyspark.sql import SparkSession

for _name in ("spark", "sc"):
    if _name in globals():
        globals()[_name] = None

try:
    from pyspark.sql.context import SQLContext
    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None
except Exception as exc:
    print(f"clearSession ignoré : {exc}")

print("Variables Spark remises à zéro. Relancez Exercice 1.")


Variables Spark remises à zéro. Relancez Exercice 1.


In [44]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

"""
Exercice 1 :
------------
Initialiser une session Spark en assignant :
- un nombre de partitions
- une taille de mémoire
- en supprimant la trace de la console
"""


def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def _reset_spark_locals() -> None:
    global spark, sc
    spark = None
    sc = None


def _create_spark_session():
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception:
            pass
    else:
        _reset_spark_locals()

    _clear_spark_session_registry()

    _reset_spark_locals()

    spark = (
        SparkSession.builder
        # [Réponse]
        .appName(APP_NAME)
        .master(MASTER)
        .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    sc = spark.sparkContext
    sc.setLogLevel("WARN")
    return spark, sc


if not _spark_is_alive():
    spark, sc = _create_spark_session()
    print(f"[Spark] session créée — version {spark.version}")
else:
    sc = spark.sparkContext
    sc.setLogLevel("WARN")
    print(f"[Spark] session réutilisée — version {spark.version}")

print(f"Spark version    : {spark.version}")
print(f"Python version   : {sc.pythonVer}")
print(f"Master           : {sc.master}")
print(f"Coeurs detectes  : {sc._jsc.sc().defaultParallelism()}")
print(f"\nSpark UI disponible sur : http://localhost:4040")


[Spark] session créée — version 3.5.9
Spark version    : 3.5.9
Python version   : 3.12
Master           : local[*]
Coeurs detectes  : 10

Spark UI disponible sur : http://localhost:4040


### Spark UI — http://localhost:4040

![Capture de la Spark UI (onglet Jobs)](localhost-4040.png)

> **Spark UI** : ouvrez l'adresse affichee ci-dessus dans votre navigateur. Vous trouverez
> l'onglet **Jobs** (vide pour l'instant), **Stages**, **Storage** et **Environment**.
> Gardez-le ouvert en permanence pendant cette journee.

### Structure d'un cluster Spark (meme en local)

```
SparkSession (driver)
  |
  +-- SparkContext
        |
        +-- Executor 1 (Thread pool)
        |     +-- Task 1-1
        |     +-- Task 1-2
        +-- Executor 2 (Thread pool)
              +-- Task 2-1
              +-- Task 2-2
```

En mode `local[*]`, le driver et les executeurs partagent le meme processus JVM.
Chaque coeur logique correspond a un *slot* d'execution de taches.


---
## 1.2 Chargement des donnees brutes avec `sc.textFile()`

Le jeu de donnees brut est le fichier `historique_stations.csv`. Chaque ligne decrit
l'etat d'une station a un instant donne. Voici le format :

```
date,capacity,available_mechanical,available_electrical,station_name,station_geo,operative
2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
...
```

L'API `sc.textFile()` retourne un **RDD de chaines de caracteres** -- une ligne du fichier
par element. Aucune interpretation du contenu n'est effectuee a ce stade.


In [45]:
"""
Exercice 2 :
------------
Charger les ficbiers de log des Velib
sc.textFile() accepte un chemin vers un fichier, un repertoire, ou un glob
"""
# [Réponse]
print("Chargement des ficbiers de log des Velib ")
raw_rdd = sc.textFile(str(HISTORIQUE_STATIONS_CSV))

"""
Exercice 3 :
------------
Compter le nombre de lignes du RDD.
count() est une ACTION -- elle declenche le calcul
"""
# [Réponse]
nb_lignes = raw_rdd.count()
print(f"Nombre de lignes : {nb_lignes:,}")

"""
Exercice 4 :
------------
Apercu des 5 premieres lignes -- take() est aussi une action
"""
# [Réponse]
print("Exercice 4 ")
for ligne in raw_rdd.take(5):
    print(ligne)


Chargement des ficbiers de log des Velib 
Nombre de lignes : 5,298,821
Exercice 4 
2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True
2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True
2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True


### Observations

- `sc.textFile()` est **instantane** : il ne lit pas encore les fichiers. Il cree
  seulement un *plan de calcul* (un RDD logique).
- `count()` declenche la lecture et le comptage -- c'est la premiere **action**.
- Les fichiers sont decoupes en **partitions** (une par bloc HDFS ou par fichier local).
  Le nombre de partitions determine le parallelisme maximal.

Verifions le nombre de partitions :


In [46]:
print(f"Nombre de partitions : {raw_rdd.getNumPartitions()}")

# En mode local, Spark cree typiquement une partition par coeur,
# ou une par fichier si le nombre de fichiers est superieur au nombre de coeurs.
# On peut forcer un repartitionnement :
raw_rdd_8p = raw_rdd.repartition(8)
print(f"Apres repartition(8) : {raw_rdd_8p.getNumPartitions()} partitions")
# Note : repartition() est une transformation -- elle ne s'execute pas encore.


Nombre de partitions : 12
Apres repartition(8) : 8 partitions


---
## 1.3 Transformations elementaires : `map`, `filter`

Un RDD est une collection **immuable** et **paresseuse** : les transformations ne
s'executent qu'au moment ou une action les requiert.

### Separation de l'en-tete

La premiere ligne de chaque fichier est un en-tete. Il faut l'eliminer avant de parser
les donnees.


In [47]:
"""
Exercice 5 :
------------
Recuperer l'en-tete pour connaitre les colonnes
Il existe une fonction spéciale pour cela
"""
# [Réponse]
entete = raw_rdd.first()   # action : renvoie le 1er element du RDD
print(entete)

# historique_stations.csv n'a pas de ligne d'en-tete : colonnes documentees
colonnes = [
    "horodatage", "capacite", "velos_meca", "velos_elec",
    "nom_station", "coordonnees", "operative",
]
print(colonnes)
"""
Exercice 6 :
------------
Filtrer les lignes d'en-tete (toutes les lignes identiques a la premiere)
filter() retourne un nouveau RDD contenant uniquement les elements
pour lesquels la fonction retourne True

1) Qu'est-ce qui s'affiche à la sortie de `filter` ?
2) Comment affijhcer le nombre de lignes de donnees ?
"""
# [Réponse]
data_rdd = raw_rdd.filter(lambda line: line != entete)

# 1) filter() est paresseux : affiche une reference RDD, pas les lignes
print(data_rdd)

# 2) count() est une action qui declenche le calcul
print(f"Lignes de donnees : {data_rdd.count():,}")
# Lignes de donnees : 5,298,820 : 5,3 millions de lignes de données.


2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
['horodatage', 'capacite', 'velos_meca', 'velos_elec', 'nom_station', 'coordonnees', 'operative']
PythonRDD[158] at RDD at PythonRDD.scala:53
Lignes de donnees : 5,298,820


### Parsing des lignes avec `map()`

`map(f)` applique la fonction `f` a chaque element du RDD et retourne un nouveau RDD
de meme longueur contenant les resultats. C'est une transformation *un-pour-un*.


In [48]:
import csv
from datetime import datetime, timezone

COLONNES = [
    "horodatage", "capacite", "velos_meca", "velos_elec",
    "nom_station", "coordonnees", "operative",
]

"""
Exercice 7  :
------------
"""
#qui sert à transformer une ligne brute d'un fichier CSV Vélib'
#  en dictionnaire exploitable.
#line: str : la fonction attend un texte (str) en entrée.
#-> dict | None : la fonction peut retourner :
#   un dict (dictionnaire Python) si la ligne est correcte ;
#   None si la ligne est invalide.

#"Parser" signifie :
#Lire une donnée brute et la convertir dans une structure utilisable.

def parse_ligne(line: str) -> dict | None:
    """Parse une ligne CSV du fichier Velib' brut.

    Args:
        line: Chaine brute separee par des virgules (champs geo entre guillemets).

    Returns:
        Dictionnaire avec les champs types, ou None si la ligne est malformee.
    """
    try:
        champs = next(csv.reader([line]))
        if len(champs) != 7:
            return None
        horodatage, capacite, velos_meca, velos_elec, nom_station, coordonnees, operative = champs
        cap = int(capacite)
        meca = int(velos_meca)
        elec = int(velos_elec)
        return {
            "horodatage": horodatage,
            "capacite": cap,
            "velos_meca": meca,
            "velos_elec": elec,
            "bornettes_libres": cap - meca - elec,
            "nom_station": nom_station,
            "coordonnees": coordonnees,
            "operative": operative.lower() == "true",
        }
    except (ValueError, StopIteration):
        return None

# Appliquer le parsing au jeu de donnees
parsed_rdd = data_rdd.map(parse_ligne)

# Supprimer les lignes malformees (None)
clean_rdd = parsed_rdd.filter(lambda x: x is not None)

# Compter les lignes
print(f"Lignes parsees : {clean_rdd.count():,}")
#:, sert à ajouter des séparateurs de milliers.

# Afficher les deux premieres
for row in clean_rdd.take(2):
    print(row)

Lignes parsees : 5,298,820
{'horodatage': '2020-11-26T12:59Z', 'capacite': 55, 'velos_meca': 23, 'velos_elec': 4, 'bornettes_libres': 28, 'nom_station': 'André Mazet - Saint-André des Arts', 'coordonnees': '48.85376,2.33910', 'operative': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 20, 'velos_meca': 0, 'velos_elec': 0, 'bornettes_libres': 20, 'nom_station': 'Charonne - Robert et Sonia Delauney', 'coordonnees': '48.85591,2.39257', 'operative': True}


> **Examen du Spark UI** : allez dans l'onglet **Jobs**. Vous devriez voir deux jobs
> termines. Cliquez sur le dernier (le `count()`). Dans la vue **Stages**, observez :
>
> - Le nombre de taches (*tasks*) -- une par partition.
> - Le temps passe dans chaque tache.
> - La colonne **Input** : la quantite de donnees lues par chaque tache.
>
> Notez que les deux `count()` que nous avons declenches ont relu les fichiers
> depuis le disque a chaque fois. C'est le comportement par defaut -- nous verrons
> comment l'eviter avec `.cache()`.


---
## 1.4 Evaluation paresseuse : transformations vs actions

C'est le concept le plus important de Spark. Toutes les **transformations** sont
**paresseuses** : elles construisent un plan de calcul (un DAG), mais ne font rien.
Seules les **actions** declenchent l'execution.

| Transformations (paresseuses) | Actions (declenchent le calcul) |
|-------------------------------|----------------------------------|
| `map()`, `filter()`, `flatMap()` | `count()`, `collect()`, `take()` |
| `groupByKey()`, `reduceByKey()` | `first()`, `top()`, `reduce()` |
| `join()`, `union()`, `distinct()` | `saveAsTextFile()`, `foreach()` |
| `repartition()`, `coalesce()` | `countByValue()`, `countByKey()` |

Illustrons ce comportement :


In [49]:
import time

# --- Construction du plan (instantane) ---
t0 = time.perf_counter()

"""
Exercice 8 :
-----------
On veut effectuer un calcul en trois étapes :
1) Relever le nombre de stations non vides
2) Calculer le nombre de vélos et le taux d'occupation
3) Ne conserver les lignes avec taux d'occupation de moins de 10%
"""
# [Réponse]
# 1) Stations non vides : capacite > 0 et au moins un velo disponible
step1 = clean_rdd.filter(
    lambda r: r["capacite"] > 0 and (r["velos_meca"] + r["velos_elec"]) > 0
)

# 2) Ajouter velos_total et taux_occupation
def ajouter_taux(r):
    cap = r["capacite"]
    velos = r["velos_meca"] + r["velos_elec"]
    return {
        **r,
        "velos_total": velos,
        "taux_occupation": (cap - r["bornettes_libres"]) / cap,
    }

step2 = step1.map(ajouter_taux)

# 3) Ne conserver que les lignes avec taux d'occupation < 10 %
step3 = step2.filter(lambda r: r["taux_occupation"] < 0.10)

# Combien de temps a pris cette opération ?
t1 = time.perf_counter()
print(f"Construction du plan (3 transformations) : {(t1-t0)*1000:.1f} ms")

# --- Declenchement par une action ---
t2 = time.perf_counter()
"""
Exercice 9 :
------------
Compter le nombre de stations
"""
# [Réponse]
n = step3.count()
t3 = time.perf_counter()

# Combien de temps a pris cette opération
print(f"Execution de count() (lecture + calcul) : {(t3-t2):.2f} s")
print(f"Lignes avec taux_occupation valide : {n:,}")


Construction du plan (3 transformations) : 0.1 ms
Execution de count() (lecture + calcul) : 2.07 s
Lignes avec taux_occupation valide : 690,858


### Le DAG (Directed Acyclic Graph)

Spark traduit votre chaine de transformations en un graphe oriente acyclique.
Ce graphe est optimise avant l'execution par le **Catalyst optimizer** (pour les
DataFrames) ou execute tel quel (pour les RDD).

```
sc.textFile()  -->  filter(header)  -->  map(parse)  -->  filter(None)
    -->  filter(capacite > 0)  -->  map(taux)  -->  filter(valide)
    -->  count()  [ACTION]
```

> **Spark UI -> Jobs -> dernier job -> DAG Visualization** : vous voyez exactement
> ce graphe. Les boites bleues sont les **stages** (separees par des shuffles).
> Une fleche grise entre deux stages indique un shuffle (transfert de donnees
> entre executeurs).


In [50]:
# On va reutiliser step3 plusieurs fois -> bonne occasion de le mettre en cache
# .cache() est equivalent a .persist(StorageLevel.MEMORY_ONLY)
step3.cache()

# Premier acces : lecture disque + mise en cache
t0 = time.perf_counter()
n1 = step3.count()
t1 = time.perf_counter()
print(f"Premier count() (lecture disque) : {t1-t0:.2f} s -- {n1:,} lignes")

# Deuxieme acces : depuis le cache en memoire
t2 = time.perf_counter()
n2 = step3.count()
t3 = time.perf_counter()
print(f"Deuxieme count() (depuis cache)  : {t3-t2:.2f} s -- {n2:,} lignes")
#.2f nbre de chiffre apres la virgule

print(f"\nGain du cache : x{((t1-t0)/(t3-t2)):.1f}")
print("\nAllez dans Spark UI -> Storage pour voir le RDD en cache.")


Premier count() (lecture disque) : 2.09 s -- 690,858 lignes
Deuxieme count() (depuis cache)  : 0.12 s -- 690,858 lignes

Gain du cache : x17.7

Allez dans Spark UI -> Storage pour voir le RDD en cache.


---
## 1.5 Agregations avec paires cle-valeur : `reduceByKey`

L'une des operations les plus importantes de l'API RDD est `reduceByKey()`. Elle
regroupe les elements qui partagent la meme cle et combine leurs valeurs avec une
fonction associative.

### Objectif : top 10 des stations par nombre de snapshots

Nous voulons savoir quelles stations apparaissent le plus souvent dans notre historique
(indicateur indirect d'une bonne couverture de collecte).


In [51]:
"""
Exercice 10 :
------------
"""
# Etape 1 : transformer chaque enregistrement en paire (cle, valeur)
# Ici : (nom_station, 1) -- on compte une occurrence par ligne
paires_rdd = step3.map(lambda r: (r["nom_station"], 1))

# Apercu
print("Exemples de paires :")
for p in paires_rdd.take(5):
    print(" ", p)


Exemples de paires :
  ('Toudouze - Clauzel', 1)
  ('Alibert - Jemmapes', 1)
  ('Cassini - Denfert-Rochereau', 1)
  ('Lacépède - Monge', 1)
  ('Morillons - Dantzig', 1)


26/07/21 14:08:33 WARN BlockManager: Task 785 already completed, not releasing lock for rdd_163_0


In [52]:
# Etape 2 : sommer les occurrences par station
# reduceByKey(f) applique f a toutes les valeurs qui ont la meme cle,
# en garantissant que f est appliquee localement avant le shuffle (combinaison locale)
comptage_rdd = paires_rdd.reduceByKey(lambda a, b: a + b)

# Etape 3 : trier par nombre decroissant et prendre le top 10
top10_rdd = comptage_rdd.sortBy(lambda x: x[1], ascending=False)

# Etape 4 : Prendre les dix premieres stations
top10 = top10_rdd.take(10)

print("Top 10 des stations par nombre de snapshots :")
print(f"{'Station':<40} {'Snapshots':>12}")
print("-" * 54)
for nom_station, count in top10:
    print(f"{nom_station:<40} {count:>12,}")


Top 10 des stations par nombre de snapshots :
Station                                     Snapshots
------------------------------------------------------
Chateaubriand - Friedland                       2,557
Alexander Fleming - Belvédère                   2,464
Cambrai - Benjamin Constant                     2,423
Bois de Vincennes - Porte Jaune                 2,381
Square du Théâtre du Garde-Chasse               2,262
Université Paris Dauphine                       2,232
Crevaux - Bugeaud                               2,170
Hoche - Tilsitt                                 2,116
Belleville - Pyrénées                           2,055
Faustin Hélie - Desbordes-Valmore               2,022


### `reduceByKey` vs `groupByKey` : une difference critique

`groupByKey()` regroupe *toutes* les valeurs en memoire avant de les reduire.
Sur de gros volumes, cela peut provoquer des `OutOfMemoryError`.
`reduceByKey()` effectue une **combinaison locale** avant le shuffle, ce qui reduit
drastiquement le volume de donnees transferees.

```
groupByKey()   : SHUFFLE toutes les valeurs -> reduire
reduceByKey()  : reduire localement -> SHUFFLE resultats partiels -> reduire
```

Regle pratique : **ne jamais utiliser `groupByKey()` quand `reduceByKey()` suffit**.


In [53]:
# Illustration de la difference de performance
# (Sur les petits volumes du cours, l'ecart peut etre faible -- il explose a l'echelle)

"""
Exercice 12 :
------------
Comparaison du temps d'execution de groupByKey() et reduceByKey() pour compter les occurrences par clef
"""
# [Réponse]
# --- groupByKey : shuffle de toutes les valeurs, puis somme ---
t0 = time.perf_counter()
paires_rdd.groupByKey().mapValues(sum).count()
t_group = time.perf_counter() - t0

# --- reduceByKey : combinaison locale avant shuffle ---
t0 = time.perf_counter()
paires_rdd.reduceByKey(lambda a, b: a + b).count()
t_reduce = time.perf_counter() - t0

print(f"groupByKey + mapValues(sum) : {t_group:.2f} s")
print(f"reduceByKey                 : {t_reduce:.2f} s")
print(f"Rapport                     : x{t_group/t_reduce:.1f}")


groupByKey + mapValues(sum) : 0.30 s
reduceByKey                 : 0.29 s
Rapport                     : x1.0


---
## 1.6 `flatMap` et `join` entre deux RDD

### `flatMap` : une entree, zero ou plusieurs sorties

Contrairement a `map` (toujours une sortie par entree), `flatMap` peut retourner
un nombre quelconque d'elements pour chaque element d'entree. C'est utile pour
"eclater" des structures imbriquees.


In [54]:
# Exemple pedagogique : a partir de chaque snapshot, on veut produire
# autant de paires (heure_de_la_journee, taux_occupation) que necessaire
# pour alimenter une distribution horaire.

def extraire_heure_et_taux(record: dict) -> list[tuple]:
    """Extrait l'heure ISO et le taux d'occupation.

    Args:
        record: Dictionnaire d'un snapshot Velib'.

    Returns:
        Liste de paires (heure_int, taux_occupation) ou liste vide si erreur.
    """
    try:
        # horodatage ex. "2020-11-26T12:59Z" -> heure = 12
        heure = int(record["horodatage"][11:13])
        return [(heure, record["taux_occupation"])]
    except (KeyError, ValueError, IndexError):
        return []

# Question : Quelle est la forme du résultat `heure_taux_rdd` ?
# RDD[(int, float)]  -- une paire (heure, taux) par snapshot
heure_taux_rdd = step3.flatMap(extraire_heure_et_taux)

"""
Exercice 13 :
-------------
Calculer la moyenne du taux d'occupation par heure de la journee
Astuce : on somme (taux, 1) puis on divise
"""
# [Réponse]
somme_count_rdd = heure_taux_rdd.map(lambda x: (x[0], (x[1], 1)))
agregat_rdd     = somme_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
moyenne_rdd     = agregat_rdd.mapValues(lambda sc: sc[0] / sc[1])
profil_horaire  = moyenne_rdd.sortByKey().collect()

print("Taux d'occupation moyen par heure de la journee (toutes stations)")
print(f"{'Heure':<8} {'Taux moyen':>12}")
print("-" * 22)
for heure, taux in profil_horaire:
    barre = "#" * int(taux * 30)
    print(f"  {heure:02d}h    {taux:.4f}  {barre}")


Taux d'occupation moyen par heure de la journee (toutes stations)
Heure      Taux moyen
----------------------
  00h    0.0620  #
  01h    0.0617  #
  02h    0.0618  #
  03h    0.0621  #
  04h    0.0621  #
  05h    0.0625  #
  06h    0.0626  #
  07h    0.0611  #
  08h    0.0596  #
  09h    0.0593  #
  10h    0.0594  #
  11h    0.0592  #
  12h    0.0593  #
  13h    0.0592  #
  14h    0.0590  #
  15h    0.0592  #
  16h    0.0600  #
  17h    0.0608  #
  18h    0.0606  #
  19h    0.0606  #
  20h    0.0612  #
  21h    0.0616  #
  22h    0.0615  #
  23h    0.0616  #


### `join` entre deux RDD

On peut joindre deux RDD de paires `(cle, valeur)` comme on joinderait deux tables.


In [55]:
"""
Exercice 14 :
-------------
"""
# [Réponse]
# RDD des stations avec leur nom : (nom_station, nom_station)
noms_rdd = step3.map(lambda r: (r["nom_station"], r["nom_station"])).distinct()

# RDD du total de velos par station : (nom_station, total_velos_disponibles)
velos_rdd = step3.map(
    lambda r: (r["nom_station"], r["velos_total"])
).reduceByKey(lambda a, b: a + b)

# Join interne
joint_rdd = noms_rdd.join(velos_rdd)
# Resultat : (nom_station, (nom_station, total_velos))

# Top 5 des stations par volume de velos disponibles (cumule)
top5 = joint_rdd.sortBy(lambda x: x[1][1], ascending=False).take(5)

print("Top 5 des stations par volume cumule de velos disponibles :")
print(f"  {'Station':<40} {'Velos cumules':>14}")
#{nom:<40} → nom à gauche, 40 caractères


print("  " + "-" * 56)
for nom, (_, total) in top5:
    print(f"  {nom:<40} {total:>14,}")
    #{total:>14,} → nombre à droite, 14 caractères, avec séparateur de milliers (,)


Top 5 des stations par volume cumule de velos disponibles :
  Station                                   Velos cumules
  --------------------------------------------------------
  Université Paris Dauphine                         7,945
  Grande Armée - Brunel                             7,493
  Cambrai - Benjamin Constant                       6,807
  Mozart - Jasmin                                   6,084
  Porte Maillot - Pereire                           6,043


---
## 1.7 Comparaison Pandas vs Spark

Reprenons le calcul "top 10 des stations par nombre de snapshots" en Pandas et
comparons les temps et la demarche.


In [56]:
import pandas as pd

COLONNES_HIST = [
    "horodatage", "capacite", "velos_meca", "velos_elec",
    "nom_station", "coordonnees", "operative",
]

# --- Version Pandas ---
t0 = time.perf_counter()

df_pandas = pd.read_csv(
    HISTORIQUE_STATIONS_CSV,
    header=None,
    names=COLONNES_HIST,
)
top10_pandas = (
    df_pandas["nom_station"].value_counts().head(10)
)
t_pandas = time.perf_counter() - t0

# --- Version Spark (deja calculee) ---
t0 = time.perf_counter()
top10_spark = top10_rdd.take(10)
t_spark = time.perf_counter() - t0   # depuis le cache

print(f"Pandas  : {t_pandas:.2f} s  --  {len(df_pandas):,} lignes chargees")
print(f"Spark   : {t_spark:.2f} s  --  depuis le cache RDD")
print()
print("Comparaison des resultats (top 5) :")
print(f"{'Station':>10}  {'Pandas':>12}  {'Spark':>12}")
for i in range(5):
    p_id, p_cnt = top10_pandas.index[i], top10_pandas.iloc[i]
    s_id, s_cnt = top10_spark[i]
    print(f"{p_id:>10}  {p_cnt:>12,}  {s_cnt:>12,}")


Pandas  : 1.72 s  --  5,298,821 lignes chargees
Spark   : 0.03 s  --  depuis le cache RDD

Comparaison des resultats (top 5) :
   Station        Pandas         Spark
Château - République         7,586         2,557
Place de l'Eglise         7,586         2,464
Université         7,586         2,423
Place de l'Europe         7,586         2,381
Benjamin Godard - Victor Hugo         3,793         2,262


In [ ]:
# --- Analyse de la memoire ---
mem_pandas_mb = df_pandas.memory_usage(deep=True).sum() / 1_048_576
print(f"\nMemoire occupee par le DataFrame Pandas  : {mem_pandas_mb:.1f} MB")
print(f"Memoire disponible pour le reste de l'OS : limitee")
print()
print("Conclusion :")
print("  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)")
print("  - Spark devient avantageux quand :")
print("    * Le volume depasse la RAM disponible")
print("    * Le calcul peut etre parallelise sur un cluster")
print("    * On combine batch + streaming + ML dans le meme pipeline")
print("    * On travaille en equipe sur un cluster partage")



Memoire occupee par le DataFrame Pandas  : 1183.2 MB
Memoire disponible pour le reste de l'OS : limitee

Conclusion :
  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)
  - Spark devient avantageux quand :
    * Le volume depasse la RAM disponible
    * Le calcul peut etre parallelise sur un cluster
    * On combine batch + streaming + ML dans le meme pipeline
    * On travaille en equipe sur un cluster partage


26/07/21 17:02:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 955165 ms exceeds timeout 120000 ms
26/07/21 17:02:33 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/21 17:02:33 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$